In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install google-generativeai datasets pandas scipy

In [1]:
import json
import google.generativeai as genai
import time
import os
from tqdm import tqdm

# --- CONFIGURATION ---
GOOGLE_API_KEY = "<YOUR_GOOGLE_API_KEY>" 
INPUT_FILE = 'geralt_big_five_complete.json'
OUTPUT_FILE = 'train_sft_gemini.jsonl'

# Safety sleep to avoid "429 Too Many Requests"
# Free Tier: Use 4 seconds. Paid Tier: Use 0.5 seconds.
SLEEP_SECONDS = 2 

# --- SETUP ---
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-2.5-flash')

system_prompt = (
    "You are Geralt of Rivia. You are a Witcher—a professional monster slayer. "
    "You are stoic, cynical, and laconic. You possess low Extraversion and low Agreeableness. "
    "You do not offer unsolicited help, and you speak in short, direct sentences."
)

# 1. Load Data
with open(INPUT_FILE, 'r') as f:
    raw_data = json.load(f)
    # Target the list inside the JSON
    all_items = raw_data['results']

# 2. Check for existing progress (Resume Logic)
processed_dialogues = set()
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r') as f:
        for line in f:
            try:
                data = json.loads(line)
                # We assume the assistant's content is unique enough to track
                # or we track by index if available. Tracking by content for now:
                processed_dialogues.add(data['messages'][2]['content'])
            except:
                pass
    print(f"Found {len(processed_dialogues)} lines already processed. Resuming...")

# 3. Processing Loop
print(f"Starting generation for {len(all_items)} total lines...")

# Open in 'append' mode ('a') so we save as we go
with open(OUTPUT_FILE, 'a') as f_out:
    
    for item in tqdm(all_items):
        geralt_line = item['dialogue']
        
        # Skip if already done
        if geralt_line in processed_dialogues:
            continue

        # Generate Context
        prompt = f"""
        I have a line of dialogue spoken by Geralt of Rivia (The Witcher).
        Write a plausible USER prompt (a question or statement from an NPC) that would cause Geralt to say this line.
        
        Geralt's Line: "{geralt_line}"
        
        Return ONLY the user prompt. No quotes.
        """
        
        try:
            response = model.generate_content(prompt)
            user_context = response.text.strip()
            
            # Create Entry
            entry = {
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_context},
                    {"role": "assistant", "content": geralt_line}
                ]
            }
            
            # Write immediately to file
            json.dump(entry, f_out)
            f_out.write('\n')
            f_out.flush() # Force save to disk
            
            # Respect API limits
            time.sleep(SLEEP_SECONDS)
            
        except Exception as e:
            print(f"\nError processing line: {geralt_line[:20]}... | {e}")
            time.sleep(10) # Wait longer if error (cooldown)

print(f"\nDone! Data saved to {OUTPUT_FILE}")

Found 3717 lines already processed. Resuming...
Starting generation for 5940 total lines...


100%|██████████| 5940/5940 [5:35:11<00:00,  3.39s/it]  


Done! Data saved to train_sft_gemini.jsonl
